# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs (all using their `@id` values).

In [ ]:
# List all available record sets by their @id
print('Record Sets (@id):')
for recordset in dataset.record_sets:
    print(f"- {recordset['@id']}: {recordset.get('name', '')}")

# For each record set, list their fields and columns by @id
for recordset in dataset.record_sets:
    print(f"\nFields in RecordSet '{recordset['@id']}' ({recordset.get('name','')}):")
    fields = recordset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"  - {field['@id']} ({field.get('name','')})")
        # If this field maps to a column, print columns
        column = field.get('column')
        if column:
            if isinstance(column, dict):
                columns = [column]
            else:
                columns = column
            for col in columns:
                col_id = col if isinstance(col, str) else col.get('@id', str(col))
                print(f"    * Column: {col_id}")
    

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Identify the record set(s) to extract (using their @id)
# We'll extract all listed record sets
record_sets_ids = [rset['@id'] for rset in dataset.record_sets]
dataframes = {}

for rset_id in record_sets_ids:
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df
    print(f"Loaded {len(df)} records from RecordSet {rset_id}, columns: {df.columns.tolist()}")

# For the main record set, display the first few rows
if record_sets_ids:
    main_record_set_id = record_sets_ids[0]  # Use the first as an example; modify as needed
    print(f"\nPreview of records from RecordSet {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing, and grouping. We'll select numeric fields by `@id` as observed above.

In [ ]:
# Select a numeric field for analysis (replace with appropriate @id based on overview)
# Example: suppose '@id': 'Age_at_SecondCRC' in the main record set
record_set_id = main_record_set_id

# Replace with the correct numeric field @id from your dataset
numeric_field_id = None
for col in dataframes[record_set_id].columns:
    if 'age' in col.lower():
        numeric_field_id = col
        break

if numeric_field_id:
    threshold = 60  # Example threshold for age
    filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Pick a categorical/group field (replace with actual @id)
    # Example: 'Sex'
    group_field_id = None
    for col in dataframes[record_set_id].columns:
        if 'sex' in col.lower() or 'gender' in col.lower():
            group_field_id = col
            break
    
    if group_field_id in dataframes[record_set_id].columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df)
else:
    print("No suitable numeric field found for EDA. Please adjust the numeric_field_id to match your dataset.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[record_set_id][numeric_field_id].dropna(), bins=15, kde=True, color='skyblue')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=dataframes[record_set_id][group_field_id], y=dataframes[record_set_id][numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to access metadata, explore record sets and fields by `@id`, and perform exploratory analysis on the FAIR^2 dataset using `mlcroissant`. The approach generalizes to any Croissant metadata-defined dataset; simply reference record sets and fields by their `@id` for reproducibility and transparency.